# Demand vs Generation SEI Scatterplots with Climate Context

This notebook loads SEI data for both renewable generation (PV) and electricity demand, then creates scatterplots comparing daily SEI values for each model, region, and global warming level. **Points are colored by climate variable anomalies to reveal climate drivers of the SEI relationship.**

**Data Sources:**
- Generation SEI: `compute_regional_drought_masks.ipynb`
- Demand SEI: `compute_demand_sei.ipynb`
- Climate data: WRF downscaled simulations via climakitae `get_data()`

**Methodology:**
- Uses modern utilities from `src/climate_correlation.py`
- Shapefile-based regional masking for proper spatial averaging
- Climate anomalies calculated from day-of-year climatology (reference: GWL 0.8°C)
- Separate plots for each GWL (not dual-GWL comparison)
- Multiple climate variables analyzed: Temperature, Humidity, Precipitation, Wind Speed

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

# Import modern utilities from src/
import sys
sys.path.append('../../src')
from climate_correlation import (
    load_wrf_climate_data_all_simulations,
    extract_simulation_gwl_period,
    compute_regional_climate_average,
    compute_anomalies
)
from renewable_data_load import get_gwl_crossing_period, sim_name_dict

# Set plot style
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Configuration
data_dir = Path("../../data/SEI")
output_dir = Path("figures/demand_vs_generation_climate")
output_dir.mkdir(parents=True, exist_ok=True)

# Shapefile for regional masking
shapefile_path = "../../data/load_zone_shapefiles/allLoadZones.shp"

# Parameters
simulations = ["ec-earth3", "miroc6", "mpi-esm1-2-hr", "taiesm1"]
gwls = [0.8, 2.0]  # Global warming levels to analyze separately
reference_gwl = 0.8  # Reference GWL used for SEI calculation (file naming) and climate anomaly baseline
domain = 'd02'

# Resource specifications
generation_resource = 'pv'
generation_module = 'utility'
generation_variable = 'cf'

demand_resource = 'demand'
demand_module = 'reference'
demand_variable = 'load'

# Climate variables to analyze
climate_vars = {
        'Mean wind speed at 10m': {
        'short': 'wind',
        'units': 'm/s',
        'cmap': 'PuOr'
    },
    'Air Temperature at 2m': {
        'short': 'temp',
        'units': '°C',
        'cmap': 'RdBu_r'
    },
    'Relative Humidity at 2m': {
        'short': 'rh',
        'units': '%',
        'cmap': 'BrBG'
    },
    'Precipitation (total)': {
        'short': 'precip',
        'units': 'mm',
        'cmap': 'BrBG'
    },

}

## Load Data

Load SEI data for both generation and demand for all models.

In [ ]:
# Dictionary to store datasets (full timeseries)
generation_sei = {}
demand_sei = {}

for simulation in simulations:
    # Load generation SEI (full timeseries with reference_gwl)
    gen_file = data_dir / f"{generation_resource}_{generation_module}_{domain}_{generation_variable}_{simulation}_gwlref{reference_gwl}_regional_SEI.nc"
    if gen_file.exists():
        generation_sei[simulation] = xr.open_dataarray(gen_file)
        print(f"Loaded generation SEI for {simulation}: {generation_sei[simulation].dims}")
    else:
        print(f"Warning: Generation SEI file not found: {gen_file}")
    
    # Load demand SEI (full timeseries with reference_gwl)
    demand_file = data_dir / f"{demand_resource}_{demand_module}_{domain}_{demand_variable}_{simulation}_gwlref{reference_gwl}_regional_SEI.nc"
    if demand_file.exists():
        demand_sei[simulation] = xr.open_dataarray(demand_file)
        print(f"Loaded demand SEI for {simulation}: {demand_sei[simulation].dims}")
    else:
        print(f"Warning: Demand SEI file not found: {demand_file}")

print(f"\nLoaded {len(generation_sei)} generation datasets and {len(demand_sei)} demand datasets")
print(f"Data will be sliced by GWL periods: {gwls}")

In [ ]:
for sim in generation_sei:
    for r in generation_sei[sim].region.values:
        print(f"Simulation: {sim}, Region: {r}")

In [ ]:
# Rename the generation regions to match demand regions
region_name_dict = {'WECC_MTN': 'WECC-MTN', 'WECC_NW': 'WECC-NW', 'WECC_SW': 'WECC-SW',
'IID': 'IID', 'LDWP': 'LDWP', 'NCNC': 'NCNC','PG&E': 'PGE','SCE':'SCE','SDG&E':'SDGE'}

for sim in generation_sei:
    # Get current region values
    current_regions = generation_sei[sim].region.values
    
    # Build rename dict only for regions that exist in the dataset
    rename_dict = {str(r): region_name_dict[str(r)] 
                   for r in current_regions 
                   if str(r) in region_name_dict}
    
    # Only rename if there's something to rename
    if rename_dict:
        generation_sei[sim] = generation_sei[sim].assign_coords(region=[rename_dict.get(str(r), str(r)) 
                                                                         for r in current_regions])
    
    print(f"{sim}: {list(generation_sei[sim].region.values)}")

In [ ]:
# Check available regions in both datasets
if simulations[0] in generation_sei and simulations[0] in demand_sei:
    gen_regions = set(generation_sei[simulations[0]].region.values)
    demand_regions = set(demand_sei[simulations[0]].region.values)
    common_regions = sorted(gen_regions.intersection(demand_regions))
    print(f"Generation regions: {sorted(gen_regions)}")
    print(f"Demand regions: {sorted(demand_regions)}")
    print(f"Common regions: {common_regions}")
else:
    common_regions = []

## Load Climate Data

Load WRF climate variables for all simulations and compute regional averages using shapefile-based masking. Climate anomalies will be computed relative to the reference GWL period (0.8°C) on a day-of-year climatology basis.

In [ ]:
# Load climate data for all variables and simulations
# Structure: regional_climate[simulation][gwl][var_short][region] = anomaly_timeseries

regional_climate = {}

for var_name, var_info in climate_vars.items():
    var_short = var_info['short']
    print(f"\n{'='*60}")
    print(f"Loading {var_name} ({var_short})...")
    print(f"{'='*60}")
    
    # Single efficient call to get all simulations at once
    climate_data_all = load_wrf_climate_data_all_simulations(
        variable=var_name,
        domain=domain,
        trim_edges=True
    )
    print(f"Loaded gridded data for all simulations: {climate_data_all.dims}")
    
    # Process each simulation
    for simulation in simulations:
        if simulation not in regional_climate:
            regional_climate[simulation] = {}
        
        print(f"\n  Processing {simulation}...")
        
        # Process each GWL
        for gwl in gwls:
            if gwl not in regional_climate[simulation]:
                regional_climate[simulation][gwl] = {}
            if var_short not in regional_climate[simulation][gwl]:
                regional_climate[simulation][gwl][var_short] = {}
            
            print(f"    GWL {gwl}...")
            
            # Extract specific simulation and GWL period
            climate_gwl = extract_simulation_gwl_period(
                climate_data=climate_data_all,
                simulation=simulation,
                gwl=gwl
            )
            
            # Compute regional averages using shapefile
            climate_regional = compute_regional_climate_average(
                climate_data=climate_gwl,
                shapefile_path=shapefile_path
            )
            
            # Get reference period data for anomaly calculation
            climate_ref = extract_simulation_gwl_period(
                climate_data=climate_data_all,
                simulation=simulation,
                gwl=reference_gwl
            )
            climate_ref_regional = compute_regional_climate_average(
                climate_data=climate_ref,
                shapefile_path=shapefile_path
            )
            
            # Compute anomalies for each region
            for region in climate_regional.region.values:
                regional_data = climate_regional.sel(region=region)
                ref_data = climate_ref_regional.sel(region=region)
                
                # Calculate day-of-year anomalies
                anomalies = compute_anomalies(
                    data=regional_data,
                    reference_data=ref_data
                )
                
                # Store in nested dictionary
                regional_climate[simulation][gwl][var_short][str(region)] = anomalies
            
            print(f"      Regions: {list(climate_regional.region.values)}")

print(f"\n{'='*60}")
print(f"Climate data loading complete!")
print(f"Variables: {list(climate_vars.keys())}")
print(f"Simulations: {simulations}")
print(f"GWLs: {gwls}")
print(f"{'='*60}")

## Create Scatterplots

For each model, region, GWL, and climate variable, create scatterplots comparing demand SEI vs generation SEI with points colored by climate anomaly.

In [ ]:
def create_sei_scatterplot_with_climate(gen_data, demand_data, climate_anom,
                                         simulation, region, gwl, var_name, var_info,
                                         save=True):
    """
    Create scatterplot comparing demand SEI vs generation SEI with climate anomaly coloring.
    
    Parameters
    ----------
    gen_data : xr.DataArray
        Generation SEI data
    demand_data : xr.DataArray
        Demand SEI data
    climate_anom : xr.DataArray
        Climate anomaly data
    simulation : str
        Model simulation name
    region : str
        Region name
    gwl : float
        Global warming level
    var_name : str
        Full climate variable name
    var_info : dict
        Dictionary with 'short', 'units', 'cmap' keys
    save : bool
        Whether to save the figure
        
    Returns
    -------
    float, int
        Correlation coefficient and sample size
    """
    # Align all data on time
    gen_aligned, demand_aligned, climate_aligned = xr.align(
        gen_data, demand_data, climate_anom, join='inner'
    )
    
    # Flatten to 1D arrays
    gen_flat = gen_aligned.values.flatten()
    demand_flat = demand_aligned.values.flatten()
    climate_flat = climate_aligned.values.flatten()
    
    # Remove NaNs
    valid_mask = (~np.isnan(gen_flat) & 
                  ~np.isnan(demand_flat) & 
                  ~np.isnan(climate_flat))
    
    gen_values = gen_flat[valid_mask]
    demand_values = demand_flat[valid_mask]
    climate_values = climate_flat[valid_mask]
    
    # Calculate correlation
    corr = np.corrcoef(gen_values, demand_values)[0, 1]
    n_samples = len(gen_values)
    
    # Create figure
    fig, ax = plt.subplots(figsize=(10, 10))
    
    # Calculate symmetric colorbar limits (equal magnitude for vmin/vmax)
    vmax_abs = max(abs(np.percentile(climate_values, 5)), 
                   abs(np.percentile(climate_values, 95)))
    
    # Create scatter plot colored by climate anomaly
    scatter = ax.scatter(gen_values, demand_values, 
                        c=climate_values, 
                        cmap=var_info['cmap'],
                        alpha=0.3, 
                        s=30,
                        edgecolors='none',
                        vmin=-vmax_abs,
                        vmax=vmax_abs)
    
    # Add colorbar
    cbar = plt.colorbar(scatter, ax=ax, pad=0.02)
    cbar.set_label(f'{var_name} Anomaly ({var_info["units"]})', 
                   fontsize=11, labelpad=10)
    
    # Add reference lines
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5, linewidth=1)
    
    # Add threshold lines (±1.28 for 80% confidence, ±1.96 for 95%)
    for threshold in [-1.96, -1.28, 1.28, 1.96]:
        ax.axhline(y=threshold, color='gray', linestyle=':', alpha=0.3, linewidth=0.8)
        ax.axvline(x=threshold, color='gray', linestyle=':', alpha=0.3, linewidth=0.8)
    
    # Labels and title
    ax.set_xlabel(f'{generation_resource.upper()} Generation SEI', fontsize=12)
    ax.set_ylabel('Demand SEI', fontsize=12)
    ax.set_title(f'{region} - {simulation} - GWL {gwl}\n' + 
                 f'Demand vs Generation SEI (colored by {var_name})\n' +
                 f'r = {corr:.3f}, n = {n_samples:,}',
                 fontsize=13, fontweight='bold')
    
    # Set equal aspect ratio and limits
    lim = max(abs(gen_values.min()), abs(gen_values.max()),
              abs(demand_values.min()), abs(demand_values.max()))
    lim = min(lim, 4)  # Cap at ±4 for readability
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_aspect('equal')
    
    # Add grid
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    if save:
        var_short = var_info['short']
        filename = f"{generation_resource}_{generation_module}_{region}_{simulation}_GWL{gwl}_{var_short}_demand_vs_generation_SEI.png"
        filepath = output_dir / filename
        plt.savefig(filepath, dpi=300, bbox_inches='tight')
        print(f"  Saved: {filename}")
    
    plt.show()
    plt.close()
    
    return corr, n_samples

In [ ]:
# TEST: Generate single scatterplot for debugging
# Specify parameters
test_simulation = "taiesm1"
test_region = "SCE"  # Replace with any region from common_regions
test_gwl = 0.8
test_var_name = 'Mean wind speed at 10m'
test_var_info = climate_vars[test_var_name]
test_var_short = test_var_info['short']

# Get WRF simulation info
WRF_sim_name = sim_name_dict[test_simulation]
model = WRF_sim_name.split("_")[1]
ensemble_member = WRF_sim_name.split("_")[2]

# Get GWL period
start_year, end_year = get_gwl_crossing_period(model, ensemble_member, test_gwl)

# Select data for specific region
gen_data_full = generation_sei[test_simulation].sel(region=test_region)
demand_data_full = demand_sei[test_simulation].sel(region=test_region)

# Slice SEI data for GWL period
gen_data = gen_data_full.sel(time=slice(f"{start_year}-01-01", f"{end_year}-12-31"))
demand_data = demand_data_full.sel(time=slice(f"{start_year}-01-01", f"{end_year}-12-31"))

# Get climate anomaly data
climate_anom = regional_climate[test_simulation][test_gwl][test_var_short][test_region]

# Create scatterplot
print(f"Creating test plot for: {test_simulation} - {test_region} - GWL {test_gwl} - {test_var_name}")
corr, n_samples = create_sei_scatterplot_with_climate(
    gen_data=gen_data,
    demand_data=demand_data,
    climate_anom=climate_anom,
    simulation=test_simulation,
    region=test_region,
    gwl=test_gwl,
    var_name=test_var_name,
    var_info=test_var_info,
    save=False  # Don't save test plot
)
print(f"Correlation: {corr:.3f}, Sample size: {n_samples:,}")

In [ ]:
# Create scatterplots for each climate variable, simulation, region, and GWL
correlations = []

for var_name, var_info in climate_vars.items():
    var_short = var_info['short']
    print(f"\n{'='*70}")
    print(f"Creating plots for {var_name}")
    print(f"{'='*70}")
    
    for simulation in simulations:
        # Check if data exists
        if simulation not in generation_sei or simulation not in demand_sei:
            print(f"Skipping {simulation} - missing SEI data")
            continue
        
        print(f"\n{simulation}:")
        
        for region in common_regions:
            print(f"  {region}:")
            
            try:
                # Get WRF simulation info
                WRF_sim_name = sim_name_dict[simulation]
                model = WRF_sim_name.split("_")[1]
                ensemble_member = WRF_sim_name.split("_")[2]
                
                # Select data for specific region (full timeseries)
                gen_data_full = generation_sei[simulation].sel(region=region)
                demand_data_full = demand_sei[simulation].sel(region=region)
                
                # Create plots for each GWL
                for gwl in gwls:
                    print(f"    GWL {gwl}...", end=" ")
                    
                    # Get GWL period
                    start_year, end_year = get_gwl_crossing_period(model, ensemble_member, gwl)
                    
                    # Slice SEI data for GWL period
                    gen_data = gen_data_full.sel(time=slice(f"{start_year}-01-01", f"{end_year}-12-31"))
                    demand_data = demand_data_full.sel(time=slice(f"{start_year}-01-01", f"{end_year}-12-31"))
                    
                    # Get climate anomaly data
                    climate_anom = regional_climate[simulation][gwl][var_short][region]
                    
                    # Create scatterplot
                    corr, n_samples = create_sei_scatterplot_with_climate(
                        gen_data=gen_data,
                        demand_data=demand_data,
                        climate_anom=climate_anom,
                        simulation=simulation,
                        region=region,
                        gwl=gwl,
                        var_name=var_name,
                        var_info=var_info,
                        save=True
                    )
                    
                    # Store correlation info
                    correlations.append({
                        'climate_var': var_name,
                        'var_short': var_short,
                        'simulation': simulation,
                        'region': region,
                        'gwl': gwl,
                        'correlation': corr,
                        'n_samples': n_samples
                    })
                
            except Exception as e:
                print(f"\n    Error processing {simulation} - {region}: {e}")
                import traceback
                traceback.print_exc()

# Create summary dataframe
corr_df = pd.DataFrame(correlations)
print("\n" + "="*80)
print("Summary of Correlations")
print("="*80)
print(corr_df.to_string(index=False))

# Save correlations to CSV
csv_path = output_dir / f"{generation_resource}_demand_vs_generation_correlations_by_climate.csv"
corr_df.to_csv(csv_path, index=False)
print(f"\nSaved correlations to: {csv_path}")

In [ ]:
gwl = 2.0
var_name = 'Air Temperature at 2m'
var_info = {'short': 'temp', 'units': '°C', 'cmap': 'RdBu_r'}
var_short = 'temp'


subset = corr_df[(corr_df['var_short'] == var_short) & (corr_df['gwl'] == gwl)]
            

pivot_corr = subset.pivot(index='region', columns='simulation', values='correlation')
                
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot_corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, cbar_kws={'label': 'Correlation'},
            linewidths=0.5, ax=ax)
ax.set_title(f'Demand vs Generation SEI Correlations\n' +
            f'{var_name} - GWL {gwl}', 
            fontsize=14, fontweight='bold', pad=20)
ax.set_xlabel('Model Simulation', fontsize=12)
ax.set_ylabel('Region', fontsize=12)
plt.tight_layout()

In [ ]:
# Create correlation heatmaps for each climate variable and GWL
if len(correlations) > 0:
    for var_name, var_info in climate_vars.items():
        var_short = var_info['short']
        
        for gwl in gwls:
            # Filter data for this variable and GWL
            subset = corr_df[(corr_df['var_short'] == var_short) & (corr_df['gwl'] == gwl)]
            
            if len(subset) > 0:
                # Create pivot table
                pivot_corr = subset.pivot(index='region', columns='simulation', values='correlation')
                
                # Create heatmap
                fig, ax = plt.subplots(figsize=(10, 6))
                sns.heatmap(pivot_corr, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
                           vmin=-1, vmax=1, cbar_kws={'label': 'Correlation'},
                           linewidths=0.5, ax=ax)
                ax.set_title(f'Demand vs Generation SEI Correlations\n' +
                           f'{var_name} - GWL {gwl}', 
                           fontsize=14, fontweight='bold', pad=20)
                ax.set_xlabel('Model Simulation', fontsize=12)
                ax.set_ylabel('Region', fontsize=12)
                plt.tight_layout()
                
                # Save figure
                heatmap_file = output_dir / f'correlation_heatmap_{generation_resource}_{generation_module}_{var_short}_GWL{gwl}.png'
                plt.savefig(heatmap_file, dpi=300, bbox_inches='tight')
                print(f"Saved: {heatmap_file}")
                plt.show()
    
    # Create summary by climate variable
    print("\n" + "="*80)
    print("Correlation Summary by Climate Variable")
    print("="*80)
    summary = corr_df.groupby(['climate_var', 'gwl'])['correlation'].agg(['mean', 'std', 'min', 'max'])
    print(summary.to_string())
else:
    print("No correlations to plot.")